In [1]:
import numpy as np 
import tensorflow as tf
from tensorflow import keras

In [2]:
text = """
The little robot walked through the quiet garden.
It watched the birds flying above the trees.
The sun was bright, and the wind moved softly.
The robot learned something new every day.
"""

print("Text length: ",len(text))
print("First 100 charcters: ")
print(text[:100])

Text length:  186
First 100 charcters: 

The little robot walked through the quiet garden.
It watched the birds flying above the trees.
The 


In [3]:
vocab = sorted(set(text))

vocab_size= len(vocab)

char_to_id = {char: idx for idx, char in enumerate(vocab)}
id_to_char = {idx: char for idx, char in enumerate(vocab)}

print("Vocabulary Size: ", vocab_size)
print("First 20 Vocabulary Characters: ",vocab[:20])

Vocabulary Size:  28
First 20 Vocabulary Characters:  ['\n', ' ', ',', '.', 'I', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o']


In [4]:
# Text -> token IDs
encoded_text = [char_to_id[char] for char in text]

# Token IDs -> text
decoded_text = "".join(id_to_char[idx] for idx in encoded_text)

print("First 30 Tokens: ",encoded_text[:30])
print("Decoded Text Matches the Originals: ",decoded_text == text)

First 30 Tokens:  [0, 5, 13, 10, 1, 16, 14, 23, 23, 16, 10, 1, 21, 19, 7, 19, 23, 1, 26, 6, 16, 15, 10, 9, 1, 23, 13, 21, 19, 24]
Decoded Text Matches the Originals:  True


In [7]:
context_length = 40

token_ids = np.array(encoded_text, dtype=np.int32)

x = []
y = []

for start in range(len(token_ids) - context_length):
    x.append(token_ids[start: start + context_length])
    y.append(token_ids[start+1: start + context_length+1])

x = np.array(x, dtype=np.int32)
y = np.array(y, dtype=np.int32)

print("X shape:", x.shape)
print("y shape:", y.shape)
print("Number of training sequences:", len(x))

X shape: (146, 40)
y shape: (146, 40)
Number of training sequences: 146


In [9]:
print("Input: ")
print("".join(id_to_char[i] for i  in x[0]))
print("\n Target: ")
print("".join(id_to_char[i] for i in y[0]))

Input: 

The little robot walked through the qui

 Target: 
The little robot walked through the quie


In [12]:
batch_size=16

# Reserve the final 10% for validation
split_idx = int(len(x) * 0.9)

x_train, x_val = x[:split_idx], x[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(min(len(x_train),1000))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val,y_val))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

for batch_X, batch_y in train_ds.take(1):
    print("Batch input shape:", batch_X.shape)
    print("Batch target shape:", batch_y.shape)
    

Batch input shape: (16, 40)
Batch target shape: (16, 40)


In [13]:
for batch_X, batch_y in train_ds.take(1):
    print("Input dtype:", batch_X.dtype)
    print("Target dtype:", batch_y.dtype)

    print("Input minimum ID:", tf.reduce_min(batch_X).numpy())
    print("Input maximum ID:", tf.reduce_max(batch_X).numpy())

    print("Vocabulary size:", vocab_size)

Input dtype: <dtype: 'int32'>
Target dtype: <dtype: 'int32'>
Input minimum ID: 0
Input maximum ID: 27
Vocabulary size: 28
